In [ ]:

# ============================================================
# PROJECT: BUILD A MINI GPT MODEL FROM SCRATCH
# FRAMEWORK: TENSORFLOW + KERAS
# LEVEL: BEGINNER / INTERMEDIATE
#
# Goal:
# Train a small GPT-style language model on sample text
# and use it to generate new text.
#
# Important:
# This is an educational, character-level GPT-style model.
# Real GPT models usually use subword tokenization, much larger
# datasets, and much larger Transformer architectures.
# ============================================================


# ------------------------------------------------------------
# STEP 1: IMPORT LIBRARIES
# ------------------------------------------------------------

import numpy as np
import tensorflow as tf

# Set random seeds to make the experiment more reproducible.
# Some hardware operations may still produce small differences.
tf.keras.utils.set_random_seed(42)


# ------------------------------------------------------------
# STEP 2: PREPARE THE TRAINING DATA
# ------------------------------------------------------------

# This is our small sample dataset.
# The model learns language patterns from these sentences.
# Add more clean sentences to improve the training experience.

text = """
artificial intelligence helps students learn new skills.
machine learning finds patterns in data.
deep learning uses neural networks to learn complex patterns.
python is a popular language for artificial intelligence.
students build projects to understand programming.
a transformer uses attention to process sequences.
attention helps a model focus on useful earlier tokens.
a language model predicts the next token from previous tokens.
gpt is a decoder only transformer language model.
good data and careful training help models learn.
students practice coding to become better developers.
data science uses statistics programming and machine learning.
neural networks contain layers with trainable weights.
a model learns by reducing its prediction error.
python tensorflow and keras help us build neural networks.
"""

# Convert text to lowercase so that uppercase and lowercase
# letters are treated as the same character.
text = text.lower().strip()

# Display the training text and its size.
print("Training text:")
print(text)
print("\nTotal characters:", len(text))


# ------------------------------------------------------------
# STEP 3: CREATE A SIMPLE TOKENIZER
# ------------------------------------------------------------

# A tokenizer converts text into numerical IDs.
#
# Example:
# "abc" might become [0, 1, 2].
#
# This example uses each character as one token.
# It is simple to understand, but real GPT models generally
# use subword tokenization.

# Find all unique characters in the training text.
vocabulary = sorted(set(text))

# Count the total number of unique characters.
vocab_size = len(vocabulary)

# Create a dictionary that converts characters into integer IDs.
char_to_id = {
    character: index
    for index, character in enumerate(vocabulary)
}

# Create a dictionary that converts integer IDs back to characters.
id_to_char = {
    index: character
    for character, index in char_to_id.items()
}

print("\nVocabulary:", vocabulary)
print("Vocabulary size:", vocab_size)

# Convert the entire training text into integer IDs.
encoded_text = np.array(
    [char_to_id[character] for character in text],
    dtype=np.int32
)

print("\nFirst 30 characters:")
print(text[:30])

print("\nTheir token IDs:")
print(encoded_text[:30])

# Convert these IDs back to characters to check the tokenizer.
decoded_text = "".join(
    id_to_char[int(token_id)]
    for token_id in encoded_text[:30]
)

print("\nDecoded text:")
print(decoded_text)


# ------------------------------------------------------------
# STEP 4: CREATE INPUT AND TARGET SEQUENCES
# ------------------------------------------------------------

# GPT learns to predict the next token.
#
# Example:
# Input:  "python"
# Target: "ython "
#
# Every input position is trained to predict the next character.
#
# For an actual sequence:
# Input:  a r t i f i c i
# Target: r t i f i c i a
#
# Notice that the target is shifted by one position.

# Maximum number of characters processed at one time.
context_length = 64

# Number of examples processed together in one batch.
batch_size = 32

# These lists will store our training examples.
all_inputs = []
all_targets = []

# Create overlapping sequences from the encoded text.
for start in range(len(encoded_text) - context_length):

    # Select a sequence of characters for the model input.
    input_sequence = encoded_text[
        start : start + context_length
    ]

    # Shift the sequence one position forward.
    # This becomes the correct next-character target.
    target_sequence = encoded_text[
        start + 1 : start + context_length + 1
    ]

    # Store both sequences.
    all_inputs.append(input_sequence)
    all_targets.append(target_sequence)

# Convert the lists into NumPy arrays.
all_inputs = np.array(all_inputs, dtype=np.int32)
all_targets = np.array(all_targets, dtype=np.int32)

print("\nInput shape:", all_inputs.shape)
print("Target shape:", all_targets.shape)

# Display the first training example.
print("\nExample input:")
print("".join(
    id_to_char[int(i)]
    for i in all_inputs[0]
))

print("\nExpected target:")
print("".join(
    id_to_char[int(i)]
    for i in all_targets[0]
))


# ------------------------------------------------------------
# STEP 5: SPLIT DATA INTO TRAINING AND VALIDATION DATA
# ------------------------------------------------------------

# Training data updates the model's weights.
# Validation data helps us check performance on held-out examples.

# Generate shuffled indices so neighboring text windows are
# not automatically assigned in their original order.
rng = np.random.default_rng(42)
indices = rng.permutation(len(all_inputs))

# Use 90% of examples for training.
split_index = int(0.9 * len(indices))

train_indices = indices[:split_index]
val_indices = indices[split_index:]

# Select training inputs and targets.
x_train = all_inputs[train_indices]
y_train = all_targets[train_indices]

# Select validation inputs and targets.
x_val = all_inputs[val_indices]
y_val = all_targets[val_indices]

print("\nTraining examples:", len(x_train))
print("Validation examples:", len(x_val))

# Convert training arrays into a TensorFlow dataset.
# shuffle() randomizes example order during training.
# batch() groups examples together.
# prefetch() helps prepare the next batch efficiently.
train_dataset = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(len(x_train))
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

# Create the validation dataset.
val_dataset = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)


# ------------------------------------------------------------
# STEP 6: BUILD THE GPT TRANSFORMER BLOCK
# ------------------------------------------------------------

# A GPT-style Transformer block contains:
#
# 1. Masked multi-head self-attention
# 2. Residual connection
# 3. Layer normalization
# 4. Feed-forward neural network
# 5. Another residual connection
# 6. Another layer normalization
#
# The causal mask prevents a position from looking at future
# characters when predicting the next character.

class GPTBlock(tf.keras.layers.Layer):

    def __init__(
        self,
        embedding_dim,
        num_heads,
        ff_dim,
        dropout_rate=0.1
    ):
        super().__init__()

        # Multi-head self-attention lets each position combine
        # information from earlier positions.
        #
        # key_dim is the dimension used by each attention head.
        self.attention = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embedding_dim // num_heads,
            dropout=dropout_rate
        )

        # Normalize the output after the attention residual connection.
        self.norm1 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6
        )

        # Feed-forward network:
        # First Dense layer expands the feature representation.
        # GELU introduces non-linearity.
        # Second Dense layer returns to embedding_dim.
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(
                ff_dim,
                activation="gelu"
            ),
            tf.keras.layers.Dense(embedding_dim)
        ])

        # Normalize the output after the feed-forward residual connection.
        self.norm2 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6
        )

        # Dropout helps reduce overfitting.
        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):

        # Masked self-attention:
        # Each position can attend only to itself and earlier positions.
        attention_output = self.attention(
            query=inputs,
            key=inputs,
            value=inputs,
            use_causal_mask=True,
            training=training
        )

        # Residual connection:
        # Add the original input to the attention output.
        x = inputs + self.dropout1(
            attention_output,
            training=training
        )

        # Layer normalization stabilizes the representation.
        x = self.norm1(x)

        # Pass each position through the feed-forward network.
        ffn_output = self.ffn(x, training=training)

        # Add another residual connection.
        x = x + self.dropout2(
            ffn_output,
            training=training
        )

        # Normalize the result.
        x = self.norm2(x)

        # Return the contextual token representations.
        return x


# ------------------------------------------------------------
# STEP 7: BUILD THE COMPLETE MINI GPT MODEL
# ------------------------------------------------------------

# The complete model follows this architecture:
#
# Token IDs
#    |
#    v
# Token Embeddings + Position Embeddings
#    |
#    v
# GPT Transformer Block 1
#    |
#    v
# GPT Transformer Block 2
#    |
#    v
# Final Layer Normalization
#    |
#    v
# Dense Output Layer
#    |
#    v
# Next-token logits
#
# Logits are scores for all possible next characters.

class MiniGPT(tf.keras.Model):

    def __init__(
        self,
        vocab_size,
        context_length,
        embedding_dim=64,
        num_heads=4,
        ff_dim=128,
        num_layers=2,
        dropout_rate=0.1
    ):
        super().__init__()

        # Store the maximum context length.
        self.context_length = context_length

        # Token embedding:
        # Convert each token ID into a trainable vector.
        self.token_embedding = tf.keras.layers.Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim
        )

        # Position embedding:
        # Learn a vector for each position in the input sequence.
        self.position_embedding = tf.keras.layers.Embedding(
            input_dim=context_length,
            output_dim=embedding_dim
        )

        # Create a stack of GPT Transformer blocks.
        self.blocks = [
            GPTBlock(
                embedding_dim=embedding_dim,
                num_heads=num_heads,
                ff_dim=ff_dim,
                dropout_rate=dropout_rate
            )
            for _ in range(num_layers)
        ]

        # Normalize the final Transformer representations.
        self.final_norm = tf.keras.layers.LayerNormalization(
            epsilon=1e-6
        )

        # Convert the hidden representation at each position
        # into one score per vocabulary character.
        #
        # We return logits rather than probabilities because
        # the loss function below expects raw logits.
        self.output_layer = tf.keras.layers.Dense(vocab_size)

    def call(self, token_ids, training=False):

        # token_ids shape:
        # (batch_size, sequence_length)
        sequence_length = tf.shape(token_ids)[1]

        # Generate position IDs: 0, 1, 2, 3, ...
        position_ids = tf.range(
            start=0,
            limit=sequence_length,
            delta=1
        )

        # Convert token IDs into dense vectors.
        token_vectors = self.token_embedding(token_ids)

        # Convert positions into dense vectors.
        position_vectors = self.position_embedding(position_ids)

        # Combine token meaning with position information.
        x = token_vectors + position_vectors

        # Pass representations through all Transformer blocks.
        for block in self.blocks:
            x = block(x, training=training)

        # Apply final normalization.
        x = self.final_norm(x)

        # Predict next-token scores at every input position.
        logits = self.output_layer(x)

        # Shape:
        # (batch_size, sequence_length, vocab_size)
        return logits


# ------------------------------------------------------------
# STEP 8: CREATE THE MODEL
# ------------------------------------------------------------

# Embedding dimension controls the size of each token vector.
embedding_dim = 64

# Number of attention heads in each Transformer block.
num_heads = 4

# Hidden size of the feed-forward network.
ff_dim = 128

# Number of Transformer blocks stacked in the model.
num_layers = 2

# Create the MiniGPT instance.
model = MiniGPT(
    vocab_size=vocab_size,
    context_length=context_length,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    ff_dim=ff_dim,
    num_layers=num_layers,
    dropout_rate=0.1
)

# Run one small batch to build the model's weights.
sample_output = model(
    tf.constant(x_train[:2]),
    training=False
)

print("\nModel output shape:", sample_output.shape)
print("Expected: (batch_size, context_length, vocab_size)")

# Display the model architecture and parameter counts.
model.summary()


# ------------------------------------------------------------
# STEP 9: COMPILE THE MODEL
# ------------------------------------------------------------

# Cross-entropy measures how well the model predicts the
# correct next character.
#
# sparse_categorical_crossentropy accepts integer class labels.
# from_logits=True means the model returns raw scores without softmax.

loss_function = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

# Adam updates the trainable weights based on calculated gradients.
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=3e-4
    ),
    loss=loss_function,
    metrics=["sparse_categorical_accuracy"]
)


# ------------------------------------------------------------
# STEP 10: TRAIN THE GPT MODEL
# ------------------------------------------------------------

# One epoch means one pass through the training dataset.
# This small corpus is for demonstration, so its generated text
# will often be repetitive or grammatically incorrect.

epochs = 100

# Stop early if validation loss stops improving.
# Restore the weights from the best validation epoch.
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

# Start training.
# Keras handles forward passes, loss calculation, backpropagation,
# gradient calculation, and optimizer weight updates.
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=epochs,
    callbacks=[early_stopping],
    verbose=1
)


# ------------------------------------------------------------
# STEP 11: GENERATE TEXT WITH THE TRAINED MODEL
# ------------------------------------------------------------

# The model generates text one character at a time.
#
# Generation process:
# 1. Encode the prompt.
# 2. Keep the latest context_length tokens.
# 3. Predict scores for the next character.
# 4. Convert scores to probabilities.
# 5. Sample the next character.
# 6. Add it to the sequence and repeat.

def generate_text(
    prompt,
    max_new_tokens=200,
    temperature=0.8
):

    # Use lowercase because our training text is lowercase.
    prompt = prompt.lower()

    # Convert known characters in the prompt into token IDs.
    # Unknown characters are skipped in this simple tokenizer.
    prompt_ids = [
        char_to_id[character]
        for character in prompt
        if character in char_to_id
    ]

    # Handle an empty or unrecognized prompt.
    if not prompt_ids:
        fallback_character = (
            " " if " " in char_to_id else vocabulary[0]
        )
        prompt_ids = [char_to_id[fallback_character]]

    # Start with the prompt tokens.
    generated_ids = list(prompt_ids)

    # Generate one new character per iteration.
    for step in range(max_new_tokens):

        # GPT can only process the most recent context_length tokens.
        context_ids = generated_ids[-context_length:]

        # Add a batch dimension:
        # (sequence_length,) becomes (1, sequence_length).
        input_tensor = tf.constant(
            [context_ids],
            dtype=tf.int32
        )

        # Run inference.
        # training=False disables dropout during generation.
        logits = model(
            input_tensor,
            training=False
        )

        # Use the final position's scores to predict the next token.
        next_token_logits = logits[0, -1, :]

        # Temperature controls randomness.
        # Lower temperature usually produces more predictable output.
        # Higher temperature generally produces more random output.
        safe_temperature = max(float(temperature), 1e-5)

        next_token_logits = (
            next_token_logits / safe_temperature
        )

        # Convert logits into a probability distribution.
        probabilities = tf.nn.softmax(next_token_logits)

        # Sample a token using the predicted probabilities.
        # Sampling allows different possible continuations.
        next_token = tf.random.categorical(
            tf.math.log(probabilities[tf.newaxis, :]),
            num_samples=1
        )[0, 0]

        # Append the sampled token to the generated sequence.
        generated_ids.append(int(next_token.numpy()))

    # Convert all token IDs, including the prompt, into characters.
    generated_text = "".join(
        id_to_char[token_id]
        for token_id in generated_ids
    )

    return generated_text


# ------------------------------------------------------------
# STEP 12: TEST THE TRAINED MODEL
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MINI GPT TEXT GENERATION")
print("=" * 60)

# Test different prompts.
prompts = [
    "artificial",
    "python",
    "a transformer"
]

# Generate text for each prompt.
for prompt in prompts:

    print("\nPrompt:", prompt)

    result = generate_text(
        prompt=prompt,
        max_new_tokens=150,
        temperature=0.8
    )

    print("Generated text:")
    print(result)


# ------------------------------------------------------------
# STEP 13: EXPERIMENT WITH TEMPERATURE
# ------------------------------------------------------------

# A low temperature makes the probability distribution sharper.
# A high temperature makes the distribution flatter.
# Compare outputs to see how randomness changes.

print("\nLOW TEMPERATURE:")
print(generate_text(
    "python",
    max_new_tokens=100,
    temperature=0.4
))

print("\nHIGH TEMPERATURE:")
print(generate_text(
    "python",
    max_new_tokens=100,
    temperature=1.2
))


# ------------------------------------------------------------
# STEP 14: SAVE THE TRAINED WEIGHTS (OPTIONAL)
# ------------------------------------------------------------

# Save the model's learned weights to a file.
# This is useful if students want to reuse the trained weights later.
model.save_weights("mini_gpt.weights.h5")

print("\nModel weights saved to mini_gpt.weights.h5")


# ============================================================
# END OF MINI GPT PROJECT
# ============================================================


Training text:
artificial intelligence helps students learn new skills.
machine learning finds patterns in data.
deep learning uses neural networks to learn complex patterns.
python is a popular language for artificial intelligence.
students build projects to understand programming.
a transformer uses attention to process sequences.
attention helps a model focus on useful earlier tokens.
a language model predicts the next token from previous tokens.
gpt is a decoder only transformer language model.
good data and careful training help models learn.
students practice coding to become better developers.
data science uses statistics programming and machine learning.
neural networks contain layers with trainable weights.
a model learns by reducing its prediction error.
python tensorflow and keras help us build neural networks.

Total characters: 818

Vocabulary: ['\n', ' ', '.', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w'

Model: "mini_gpt_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (2, 64, 64)            │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_3 (Embedding)         │ (64, 64)               │         4,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_block_2 (GPTBlock)          │ ?                      │        33,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpt_block_3 (GPTBlock)          │ ?                      │        33,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_9           │ (2, 64, 64)            │           128 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (2, 64, 28)            │         1,820 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 74,780 (292.11 KB)

 Trainable params: 74,780 (292.11 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 12s 168ms/step - loss: 3.0708 - sparse_categorical_accuracy: 0.1630 - val_loss: 2.5721 - val_sparse_categorical_accuracy: 0.2395
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 117ms/step - loss: 2.4800 - sparse_categorical_accuracy: 0.2446 - val_loss: 2.3651 - val_sparse_categorical_accuracy: 0.2558
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 121ms/step - loss: 2.3425 - sparse_categorical_accuracy: 0.2537 - val_loss: 2.2742 - val_sparse_categorical_accuracy: 0.2654
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 6s 183ms/step - loss: 2.2684 - sparse_categorical_accuracy: 0.2584 - val_loss: 2.2253 - val_sparse_categorical_accuracy: 0.2584
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 124ms/step - loss: 2.2229 - sparse_categorical_accuracy: 0.2660 - val_loss: 2.1943 - val_sparse_categorical_accuracy: 0.2673
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 124ms/step - loss: 2.1877 - sparse_categorical_accuracy: 0.2695 - val_loss: 2.1626 - val_sparse_categorical_accuracy: 0.27

In [ ]:
# Generate text from a prompt
prompt = "artificial intelligence"

# Ask the model to generate 100 new characters
result = generate_text(
    prompt=prompt,
    max_new_tokens=100,
    temperature=0.8
)

# Display the generated text
print("Input:", prompt)
print("Output:", result)

Input: artificial intelligence
Output: artificial intelligence.
students build projects to understand programming.
a transformer and patterns in data.
deep learni


In [ ]:
print(generate_text("python", max_new_tokens=100))
print(generate_text("machine learning", max_new_tokens=100))
print(generate_text("students", max_new_tokens=100))

python is a popular language for artificial intelligence.
students build projects to understand programmin
machine learning.
neural networks contain layers with trainable weights.
a model learns by reducing its prediction e
students learn new skills.
machine learning finds paterns in data.
deep learning uses neural networks to lea


In [ ]:

import numpy as np
import tensorflow as tf

# Load the same training text used to train the model.
# The vocabulary must be reconstructed in exactly the same way.
text = """
artificial intelligence helps students learn new skills.
machine learning finds patterns in data.
deep learning uses neural networks to learn complex patterns.
python is a popular language for artificial intelligence.
students build projects to understand programming.
a transformer uses attention to process sequences.
attention helps a model focus on useful earlier tokens.
a language model predicts the next token from previous tokens.
gpt is a decoder only transformer language model.
good data and careful training help models learn.
students practice coding to become better developers.
data science uses statistics programming and machine learning.
neural networks contain layers with trainable weights.
a model learns by reducing its prediction error.
python tensorflow and keras help us build neural networks.
"""

text = text.lower().strip()

# Rebuild the exact character-to-ID mapping.
vocabulary = sorted(set(text))
vocab_size = len(vocabulary)

char_to_id = {
    character: index
    for index, character in enumerate(vocabulary)
}

id_to_char = {
    index: character
    for character, index in char_to_id.items()
}

# These settings must match the training program.
context_length = 64
embedding_dim = 64
num_heads = 4
ff_dim = 128
num_layers = 2

# GPTBlock and MiniGPT must be defined above this code.
model = MiniGPT(
    vocab_size=vocab_size,
    context_length=context_length,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    ff_dim=ff_dim,
    num_layers=num_layers,
    dropout_rate=0.1
)

# Build the model variables before loading weights.
dummy_input = tf.zeros((1, context_length), dtype=tf.int32)
model(dummy_input)

# Load the weights saved by the training script.
model.load_weights("mini_gpt.weights.h5")

print("Mini GPT model loaded successfully!")

# Use your existing generate_text() function here.
# Make sure its definition is also included in this file.
print(generate_text(
    "python",
    max_new_tokens=100,
    temperature=0.8
))


Mini GPT model loaded successfully!
python is a popular language for artificial intelligence.
students build projects to understand programmin


In [ ]:
result = generate_text(
    prompt="python",
    max_new_tokens=100,
    temperature=0.3
)

print(result)

python is a popular language for artificial intelligence.
students build projects to understand programmin
